In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import os
print("libraries imported successfully.")

In [ ]:
DATA_DIR= Path("../data/raw/epl_matches_2010_2021")
csv_files=sorted(DATA_DIR.glob("*.csv"))
print("Total CSV files: ", len(csv_files))
for file in csv_files:
    print(file.name)

In [ ]:
#Checking one csv file
df=pd.read_csv(csv_files[0])
print(df.shape)
print(df.head())
print(df.columns.tolist())
print(df.dtypes)
print(df.isna().sum())
print(df.columns.str.strip())


In [ ]:
#Loading all
dataframes=[]
for file in csv_files:
    df=pd.read_csv(file)
    #Track the source CSV file for each record
    df["Source_File"]=file.stem
    dataframes.append(df)

#Combining all seasons:
raw_matches=pd.concat(dataframes, ignore_index=True)
print("Combined Dataset Shape: ", raw_matches.shape)
print(raw_matches.head())
print(f"Total Matches: {len(raw_matches)}")
print(f"Coulmns: {raw_matches.columns.tolist()}")
print(f"Data Types: {raw_matches.dtypes}")


In [ ]:
#Renaming the column names:
matches=raw_matches.rename(columns={
    "Round": "MatchWeek",
    "Date": "Date",
    "Team 1": "HomeTeam",
    "FT": "FullTimeScore",
    "Team 2": "AwayTeam"
}).copy()
print("Rows: ", matches.shape[0])
print("Columns: ", matches.shape[1])
print(matches.tail())
print(matches.sample(5, random_state=42))
print(matches.info())
matches.head(10)

In [ ]:
# Checking values and duplicates
missing_values=matches.isnull().sum()
print(missing_values)
missing_percentage=(matches.isnull().mean()*100).round(2)
print(missing_percentage)
print(matches.duplicated().sum())
duplicates=matches[matches.duplicated(keep=False)]
print(duplicates.head(20))

In [ ]:
# Check date info
matches["Date"]=matches["Date"].astype("string").str.replace(r"\s*\(P\)$", "", regex=True)
matches["Date"]=pd.to_datetime(
    matches["Date"], 
    errors="coerce",
    dayfirst=True
)
print(matches["Date"].dtype)
print("First Match: ",{matches["Date"].min()})
print("Last Match: ",{matches["Date"].max()})
print("Invalid dates: ", matches["Date"].isna().sum())
invalid_dates = matches[matches["Date"].isna()]
display(invalid_dates[["Date", "HomeTeam", "AwayTeam","FullTimeScore", "Source_File"]])

In [ ]:
#Score processing
score_parts=matches["FullTimeScore"].astype("string").str.extract(r"^\s*(\d+)\s*[-–—:]\s*(\d+)\s*$")
matches["HomeGoals"]=pd.to_numeric(
    score_parts[0],
    errors="coerce"
)
matches["AwayGoals"]=pd.to_numeric(
    score_parts[1],
    errors="coerce"
)
print(matches[["HomeTeam", "FullTimeScore", "AwayTeam", "HomeGoals", "AwayGoals"]].head(10))
print("Invalid scores:",matches[["HomeGoals", "AwayGoals"]].isna().any(axis=1).sum())

In [ ]:
#Predict whether home team win loss or draw.
conditions=[
    matches["HomeGoals"]>matches["AwayGoals"],
    matches["HomeGoals"]<matches["AwayGoals"],
    matches["HomeGoals"]==matches["AwayGoals"]
]
results=["HW", "AW", "D"]
matches["MatchResult"]=np.select(conditions, results, default=None)
print(matches[["HomeTeam", "HomeGoals","AwayGoals", "AwayTeam", "MatchResult"]].tail(10))
print(matches["MatchResult"].value_counts())
print("In percentage:", matches["MatchResult"].value_counts(normalize=True).mul(100).round(2))

In [ ]:
#Exploring teams and seasons
# No of unique teams
all_teams=pd.concat([matches["HomeTeam"], matches["AwayTeam"]])
print("Total Number of unique Teams: ", all_teams.nunique())
#Match per season file
season_matches=matches["Source_File"].value_counts().sort_index()
print(season_matches)
#Visualize:
plt.figure(figsize=(12, 5))
season_matches.plot(kind="bar")
plt.title("Matches Per Season")
plt.xlabel("Season")
plt.ylabel("Number of Matches")
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.tight_layout()
plt.show()
print(matches[["HomeGoals", "AwayGoals"]].describe())
os.makedirs("../data/processed", exist_ok=True)
matches.to_csv("../data/processed/epl_matches_stage1.csv", index=False)
print("Dataset saved successfully")

In [ ]:
#REPORT
print("Total Matches: ", len(matches))
print("Unique Teams: ", all_teams.nunique())
print("Date Range: ", matches["Date"].min(), "to", matches["Date"].max())
print("Missing Values: ", matches.isnull().sum())
print("Duplicate Rows: ", matches.duplicated().sum())
print("Invalid Scores: ", matches[["HomeGoals", "AwayGoals"]].isna().any(axis=1).sum())
print("Match Result Distribution: ", matches["MatchResult"].value_counts())